In [ ]:
import requests
from bs4 import BeautifulSoup

In [15]:
# Knowledge Base & State Representation Architect

class KnowledgeBase:
    """
    Stores the road network graph, domain rules, and helper methods
    used by the inference and search engine to evaluate the valid trailer moves.
    """

    def __init__(self):
        # Insert graph nodes and costs from UCS_Trailer.ipynb
        # Structure: { "Origin_Node": { "Destination_Node": step_cost, ... } }
        self.graph_edges = {
            "Nairobi": {
                "Thika": 45.1,
                "Nyeri": 151,
                "Naivasha": 91,
                "Narok": 143
            },

            "Thika": {
                "Embu": 92.3
            },

            "Embu": {},

            "Nyeri": {
                "Nyahururu": 104
            },

            "Nyahururu": {
                "Nakuru": 63.3
            },

            "Naivasha": {
                "Nakuru": 71.1
            },

            "Nakuru": {
                "Eldoret": 156,
                "Kericho": 107
            },

            "Eldoret": {
                "Kisumu": 138
            },

            "Kericho": {
                "Kisumu": 84
            },

            "Narok": {
                "Kisii": 163
            },

            "Kisii": {
                "Kisumu": 114
            },

            "Kisumu": {}
        }
        # Dictionary : Holds domain rules and dynamic environment states 
        self.rules = {
            "max_load_weight_tons": 25,
            "blocked_routes": set()
        }

         # Return available neighbouring locations
    def get_neighbors(self, current_node):

        neighbors = self.graph_edges.get(current_node, {})

        available_neighbors = {}

        for neighbor, distance in neighbors.items():

            route = (current_node, neighbor)

            if route not in self.rules["blocked_routes"]:
                available_neighbors[neighbor] = distance

        return available_neighbors

    # Get the distance between two connected locations
    def get_cost(self, start, destination):

        return self.graph_edges.get(start, {}).get(destination)

    # Check whether a rule is satisfied
    def check_rule(self, rule_key, value):

        if rule_key == "max_load_weight_tons":

            if value is None:
                return True

            return (
                isinstance(value, (int, float))
                and not isinstance(value, bool)
                and 0 <= value <= self.rules[rule_key]
            )

        return False

    # Add a blocked directed route
    def add_blocked_route(self, start, destination):

        if destination not in self.graph_edges.get(start, {}):
            raise ValueError("That route does not exist in the graph.")

        self.rules["blocked_routes"].add((start, destination))

    # Remove a blocked route
    def remove_blocked_route(self, start, destination):

        self.rules["blocked_routes"].discard((start, destination))


In [16]:

# Create the knowledge base
kb = KnowledgeBase()

# Display Nairobi's available neighbours
print("Nairobi neighbours:")
print(kb.get_neighbors("Nairobi"))

# Check the distance from Nairobi to Naivasha
print("\nDistance to Naivasha:")
print(kb.get_cost("Nairobi", "Naivasha"))

# Check the trailer load rule
print("\nIs a 20-ton load allowed?")
print(kb.check_rule("max_load_weight_tons", 20))

# Block the Nairobi to Naivasha route
kb.add_blocked_route("Nairobi", "Naivasha")

print("\nNairobi neighbours after blocking Naivasha:")
print(kb.get_neighbors("Nairobi"))


Nairobi neighbours:
{'Thika': 45.1, 'Nyeri': 151, 'Naivasha': 91, 'Narok': 143}

Distance to Naivasha:
91

Is a 20-ton load allowed?
True

Nairobi neighbours after blocking Naivasha:
{'Thika': 45.1, 'Nyeri': 151, 'Narok': 143}


In [7]:
def fetch_dynamic_route_conditions(self, url=None):
    """
    Fulfills PE4 requirement: Uses web scraping / API queries (where permitted)
    to dynamically pull traffic or road constraints into the knowledge base.
    """
    try:
        if url:
            response = requests.get(url, timeout=5)
            soup = BeautifulSoup(response.text, 'html.parser')
            print("Successfully queried external source for route updates.")
        else:
            print("No external URL provided; using standard knowledge base rules.")
    except Exception as e:
        print(f"Could not retrieve live data: {e}. Falling back to static rules.")

In [8]:
#Initialize Agent's knowledge base:
kb = KnowledgeBase()
print(f"Knowledge Base initialized with {len(kb.graph_edges)} base transit nodes from Practical exercise 3.")

Knowledge Base initialized with 9 base transit nodes from Practical exercise 3.


In [17]:
## Create the UCS search adapter


import heapq

def ucs_search(knowledge_base, start, goal):
    """
    Uses Uniform Cost Search to find the cheapest
    available route between two locations.
    """

    # Priority queue stores:
    # (total cost, current location, path)
    priority_queue = []

    heapq.heappush(priority_queue, (0, start, [start]))

    # Stores the cheapest known cost to each location
    best_cost = {start: 0}

    while priority_queue:

        # Get the route with the lowest total cost
        cost, current_node, path = heapq.heappop(priority_queue)

        # Ignore an outdated, more expensive route
        if cost > best_cost[current_node]:
            continue

        # Check whether we have reached our destination
        if current_node == goal:
            return path, cost

        # Get available neighbouring locations
        neighbors = knowledge_base.get_neighbors(current_node)

        for neighbor, distance in neighbors.items():

            new_cost = cost + distance

            # Update if this is a cheaper route
            if neighbor not in best_cost or new_cost < best_cost[neighbor]:

                best_cost[neighbor] = new_cost

                new_path = path + [neighbor]

                heapq.heappush(
                    priority_queue,
                    (new_cost, neighbor, new_path)
                )

    # No available route was found
    return None, float("inf")


In [18]:
## Fact-checking function


def check_facts(knowledge_base, start, goal):

    # Get all known locations
    known_locations = set(knowledge_base.graph_edges.keys())

    # Include destinations appearing in the graph
    for neighbors in knowledge_base.graph_edges.values():
        known_locations.update(neighbors.keys())

    # Check starting location
    if start not in known_locations:
        return False, f"Unknown starting location: {start}"

    # Check destination
    if goal not in known_locations:
        return False, f"Unknown destination: {goal}"

    return True, "Starting location and destination are valid."


In [19]:
## Rule-checking function


def apply_rules(knowledge_base, start, goal, load_weight=None):

    # First check whether the locations are valid
    valid, message = check_facts(
        knowledge_base, start, goal
    )

    if not valid:
        return False, message

    # Check the trailer's load capacity
    if load_weight is not None:

        if not knowledge_base.check_rule(
            "max_load_weight_tons", load_weight
        ):
            return False, "Trailer load exceeds the allowed capacity."

    return True, "All applicable rules have been satisfied."


In [20]:
# Combine everything


def infer_route(knowledge_base, start, goal, load_weight=None):

    # Step 1: Check facts and rules
    valid, message = apply_rules(
        knowledge_base, start, goal, load_weight
    )

    if not valid:
        return {
            "success": False,
            "message": message
        }

    # Step 2: Apply Uniform Cost Search
    path, total_distance = ucs_search(
        knowledge_base, start, goal
    )

    # Step 3: Check whether a route was found
    if path is None:
        return {
            "success": False,
            "message": "No available route was found."
        }

    # Step 4: Return the inferred route
    return {
        "success": True,
        "route": path,
        "total_distance": total_distance,
        "message": "Optimal route successfully identified."
    }


Test Interference Engine

In [26]:
# Test 1: Normal journey

kb1 = KnowledgeBase()
print(infer_route(kb1, "Nairobi", "Kisumu", 20))

{'success': True, 'route': ['Nairobi', 'Naivasha', 'Nakuru', 'Kericho', 'Kisumu'], 'total_distance': 353.1, 'message': 'Optimal route successfully identified.'}


In [27]:
# Test 2: Excessive load

kb2 = KnowledgeBase()
print(infer_route(kb2, "Nairobi", "Kisumu", 30))

{'success': False, 'message': 'Trailer load exceeds the allowed capacity.'}


In [28]:
# Test 3: Unknown destination

kb3 = KnowledgeBase()
print(infer_route(kb3, "Nairobi", "Atlantis", 10))

{'success': False, 'message': 'Unknown destination: Atlantis'}


In [29]:
# Test 4: Blocked route

kb4 = KnowledgeBase()
kb4.add_blocked_route("Nairobi", "Naivasha")
print(infer_route(kb4, "Nairobi", "Kisumu", 20))

{'success': True, 'route': ['Nairobi', 'Narok', 'Kisii', 'Kisumu'], 'total_distance': 420, 'message': 'Optimal route successfully identified.'}


In [30]:
# Test 5: No available route

kb5 = KnowledgeBase()

# Block all roads leaving Nairobi
kb5.add_blocked_route("Nairobi", "Thika")
kb5.add_blocked_route("Nairobi", "Nyeri")
kb5.add_blocked_route("Nairobi", "Naivasha")
kb5.add_blocked_route("Nairobi", "Narok")

result = infer_route(kb5, "Nairobi", "Kisumu", 20)

print("TEST 5: No available route")
print(result)

TEST 5: No available route
{'success': False, 'message': 'No available route was found.'}


In [31]:
# Test 6: Starting location is also the destination

kb6 = KnowledgeBase()
result = infer_route(kb6, "Nairobi", "Nairobi", 20)

print("\nTEST 6: Same starting point and destination")
print(result)


TEST 6: Same starting point and destination
{'success': True, 'route': ['Nairobi'], 'total_distance': 0, 'message': 'Optimal route successfully identified.'}



## Inference Engine and Search Adapter

The inference engine connects the Knowledge Base to the Uniform Cost Search (UCS) algorithm from Practical Exercise 3.

It uses the information stored in the Knowledge Base, including road connections, distances, trailer capacity rules and blocked routes, to determine whether a journey can be completed.

The implementation has four main stages:

1. Fact checking: The 'check_facts()' function checks whether the starting location and destination exist in the knowledge base.
2. Rule checking: The 'apply_rules()' function checks whether the trailer satisfies the maximum load capacity rule.
3. Route searching: The 'ucs_search()' function applies Uniform Cost Search to find the available route with the lowest total distance.
4. Result generation: The 'infer_route()' function combines the checks and search results and returns the route, total distance and status message.

The UCS algorithm uses a priority queue to explore the lowest-cost route first. It updates the best known cost whenever it finds a cheaper path. If a route is blocked, the Knowledge Base excludes that connection from the available neighbours.

This allows the trailer agent to reject invalid requests and identify the cheapest available route using the stored facts and rules.
